# Shrub Lists Extras — Feature Join Prototype

Join shrub targets with RAP and 3DEP feature blocks to create a first training table skeleton.

In [ ]:
import ee
import pandas as pd
import numpy as np
from pathlib import Path

ee.Authenticate()
ee.Initialize(project="shrubwise-dc-488219")

RAP_veg_yearly_10m = ee.ImageCollection("projects/rap-data-365417/assets/vegetation-cover-10m")
THREEDEP_10M = ee.ImageCollection("USGS/3DEP/10m_collection")


In [ ]:
STD_REV = Path('standardized') / 'shrubs_revised_standardized.csv'
shrubs = pd.read_csv(STD_REV)
print(shrubs.shape)
shrubs.head()


In [ ]:
def rap_shrub_cover_10m(year):
    return RAP_veg_yearly_10m.filter(ee.Filter.eq('year', year)).mosaic().select('SHR').toFloat().rename('rap_shrub_cover')

def rap_prior_minimal(year=2025):
    current = rap_shrub_cover_10m(year).rename(f'rap_shrub_{year}')
    mean = ee.ImageCollection([rap_shrub_cover_10m(y) for y in range(2018, 2026)]).mean().rename('rap_shrub_mean_2018_2025')
    return ee.Image.cat([current, mean])

def terrain_minimal():
    dem = THREEDEP_10M.mosaic().select('elevation').toFloat().rename('elevation')
    slope = ee.Terrain.slope(dem).rename('slope_deg')
    aspect = ee.Terrain.aspect(dem).rename('aspect_deg')
    aspect_rad = aspect.multiply(np.pi / 180.0)
    northness = aspect_rad.cos().rename('northness')
    eastness = aspect_rad.sin().rename('eastness')
    return ee.Image.cat([dem, slope, aspect, northness, eastness])

feature_img = ee.Image.cat([rap_prior_minimal(2025), terrain_minimal()])
print(feature_img.bandNames().getInfo())


In [ ]:
def sample_features_at_points(df, *, x_col='x', y_col='y', scale=10):
    work = df.dropna(subset=[x_col, y_col]).copy()
    rows = []
    for _, row in work.iterrows():
        pt = ee.Geometry.Point([float(row[x_col]), float(row[y_col])])
        vals = feature_img.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=pt,
            scale=scale,
            maxPixels=1e9,
            bestEffort=True
        ).getInfo()
        rec = row.to_dict()
        rec.update(vals if vals is not None else {})
        rows.append(rec)
    return pd.DataFrame(rows)

# IMPORTANT:
# This assumes x/y are lon/lat. If shrub lists are in UTM/projected coordinates,
# reproject them to lon/lat before using ee.Geometry.Point.


In [ ]:
# Example usage ONLY if x/y are lon/lat:
# joined = sample_features_at_points(shrubs.head(50), scale=10)
# joined.head()
